 CNN

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

tf.random.set_seed(42)
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")

x_train shape: (60000, 28, 28, 1)
y_train shape: (60000, 10)


Manual Search CV



In [2]:
def create_cnn_model(learning_rate=0.01, num_filters=32):
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(num_filters, kernel_size=(3, 3), activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("\n--- Manual Search Example ---")
print("\nTraining with learning_rate=0.01, num_filters=32")
model1 = create_cnn_model(learning_rate=0.01, num_filters=32)
model1.fit(x_train, y_train, epochs=1, batch_size=32, verbose=0)
loss1, acc1 = model1.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc1:.4f}")

print("\nTraining with learning_rate=0.001, num_filters=32")
model2 = create_cnn_model(learning_rate=0.001, num_filters=32)
model2.fit(x_train, y_train, epochs=1, batch_size=32, verbose=0)
loss2, acc2 = model2.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc2:.4f}")

print("\nTraining with learning_rate=0.01, num_filters=64")
model3 = create_cnn_model(learning_rate=0.01, num_filters=64)
model3.fit(x_train, y_train, epochs=1, batch_size=32, verbose=0)
loss3, acc3 = model3.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc3:.4f}")


--- Manual Search Example ---

Training with learning_rate=0.01, num_filters=32
Test Accuracy: 0.9749

Training with learning_rate=0.001, num_filters=32
Test Accuracy: 0.9791

Training with learning_rate=0.01, num_filters=64
Test Accuracy: 0.9731


Randomized Search CV



In [3]:
!pip install scikeras

In [5]:
!pip install scikit-learn==1.4.2

import scikeras
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

def build_classifier(learning_rate=0.01, num_filters=32):
    model = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(num_filters, kernel_size=(3, 3), activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("\n--- Randomized Search CV Example ---")
keras_model = KerasClassifier(model=build_classifier, verbose=0, epochs=1, batch_size=32)

param_distributions = {
    'model__learning_rate': uniform(loc=0.0001, scale=0.01),
    'model__num_filters': randint(low=16, high=64)
}

random_search = RandomizedSearchCV(
    estimator=keras_model,
    param_distributions=param_distributions,
    n_iter=5,
    cv=2,
    verbose=0,
    scoring='accuracy',
    random_state=42
)


random_search.fit(x_train[:1000], y_train[:1000])

print(f"Best parameters found: {random_search.best_params_}")
print(f"Best cross-validation accuracy: {random_search.best_score_:.4f}")

best_model = random_search.best_estimator_
loss_rand, acc_rand = best_model.model_.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy with best random search model: {acc_rand:.4f}")


--- Randomized Search CV Example ---
Best parameters found: {'model__learning_rate': np.float64(0.004558327528535912), 'model__num_filters': 38}
Best cross-validation accuracy: 0.7520
Test Accuracy with best random search model: 0.8399


Grid Search CV


In [7]:
from sklearn.model_selection import GridSearchCV

print("\n--- Grid Search CV Example ---")
keras_model_grid = KerasClassifier(model=build_classifier, verbose=0, epochs=1, batch_size=32)

param_grid = {
    'model__learning_rate': [0.01, 0.001],
    'model__num_filters': [32, 64]
}

grid_search = GridSearchCV(
    estimator=keras_model_grid,
    param_grid=param_grid,
    cv=2,
    verbose=0,
    scoring='accuracy'
)

grid_search.fit(x_train[:1000], y_train[:1000])

print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

best_model_grid = grid_search.best_estimator_
loss_grid, acc_grid = best_model_grid.model_.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy with best grid search model: {acc_grid:.4f}")


--- Grid Search CV Example ---
Best parameters found: {'model__learning_rate': 0.01, 'model__num_filters': 32}
Best cross-validation accuracy: 0.7470
Test Accuracy with best grid search model: 0.8777


In [9]:
print("--- HPO Results Summary ---")

best_manual_result = {
    'acc1': acc1,
    'acc2': acc2,
    'acc3': acc3,
    'avg_accuracy': (acc1 + acc2 + acc3) / 3
}
print(f"Manual Search CV Average Test Accuracy: {best_manual_result['avg_accuracy']:.4f}")

if 'random_search' in locals():
    print(f"Randomized Search CV Best Cross-Validation Accuracy: {random_search.best_score_:.4f}")
else:
    print("Randomized Search CV results not available (due to previous error or variable name mismatch).")

if 'grid_search' in locals():
    print(f"Grid Search CV Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")
else:
    print("Grid Search CV results not available (due to previous error or variable name mismatch).")

print("\nNote: The 'best' parameters and accuracies might vary based on the specific search space and random states.")

--- HPO Results Summary ---
Manual Search CV Average Test Accuracy: 0.9757
Randomized Search CV Best Cross-Validation Accuracy: 0.7520
Grid Search CV Best Cross-Validation Accuracy: 0.7470

Note: The 'best' parameters and accuracies might vary based on the specific search space and random states.
